# PaySim Fraud Baseline 6D

This notebook is the canonical plaintext baseline for the privacy-preserving fraud detection prototype based on PaySim.

The goal is a compact, stable, HE-friendly inference pipeline rather than state-of-the-art fraud detection. We keep the 6D base representation fixed and compare a small set of linear and low-degree polynomial models that are practical for later HE experiments.


## 1. Project Overview

We focus on a clean plaintext baseline that:

- keeps only `TRANSFER` and `CASH_OUT`
- retains all fraud rows and downsamples non-fraud rows
- uses the same canonical 6D HE-friendly base feature vector
- compares four models on the same sampled split
- exports only the artifacts needed for later HE-side inference

Models in this notebook:

- Ridge score baseline
- Logistic Regression
- LinearSVC
- Degree-2 polynomial Logistic Regression


In [1]:
from __future__ import annotations

import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import expit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.svm import LinearSVC

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")


## 2. Imports And Configuration

The configuration below fixes paths, seeds, selected transaction types, sampling ratio, and the exported artifact set. The base 6D representation and split logic remain unchanged from the current baseline.


In [2]:
PROJECT_ROOT = Path(".").resolve()
DATA_PATH = PROJECT_ROOT / "data" / "PS_20174392719_1491204439457_log.csv"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
ARCHIVE_DIR = PROJECT_ROOT / "archive"

ARTIFACT_DIR.mkdir(exist_ok=True)
ARCHIVE_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
CHUNK_SIZE = 250_000
NEG_POS_RATIO = 10
TEST_SIZE = 0.20
VAL_SIZE_WITHIN_TRAIN = 0.20
SELECTED_TYPES = ["TRANSFER", "CASH_OUT"]

NUMERIC_BASE_FEATURES = [
    "amount",
    "oldbalanceOrg",
    "oldbalanceDest",
    "deltaOrig",
    "deltaDest",
]
NUMERIC_SCALED_FEATURES = [
    "amount_scaled",
    "oldbalanceOrg_scaled",
    "oldbalanceDest_scaled",
    "deltaOrig_scaled",
    "deltaDest_scaled",
]
BINARY_FEATURES = ["is_transfer"]
FEATURE_COLUMNS = NUMERIC_SCALED_FEATURES + BINARY_FEATURES

SUMMARY_JSON_PATH = ARTIFACT_DIR / "baseline_6d_summary.json"
COMPARISON_JSON_PATH = ARTIFACT_DIR / "baseline_model_comparison_summary.json"
FEATURE_ORDER_JSON_PATH = ARTIFACT_DIR / "feature_order_6d.json"
SCALER_PICKLE_PATH = ARTIFACT_DIR / "baseline_6d_scaler.pkl"
RIDGE_PICKLE_PATH = ARTIFACT_DIR / "baseline_6d_ridge.pkl"
LOGREG_PICKLE_PATH = ARTIFACT_DIR / "baseline_6d_logreg.pkl"
LINEAR_SVC_PICKLE_PATH = ARTIFACT_DIR / "linear_svc_model.pkl"
POLY2_LOGREG_PICKLE_PATH = ARTIFACT_DIR / "poly2_logreg_model.pkl"
POLY2_TRANSFORMER_PICKLE_PATH = ARTIFACT_DIR / "poly2_transformer.pkl"
HE_FEATURES_PATH = ARTIFACT_DIR / "he_eval_scaled_features_6d.csv"
HE_META_PATH = ARTIFACT_DIR / "he_eval_meta_6d.csv"
HE_SCORES_PATH = ARTIFACT_DIR / "he_eval_plaintext_scores_6d.csv"
PREVIOUS_8D_ARTIFACT_PATH = ARCHIVE_DIR / "baseline_model_artifacts_8d.json"

assert DATA_PATH.exists(), f"Missing dataset: {DATA_PATH}"

print("Project root:", PROJECT_ROOT)
print("Dataset:", DATA_PATH)
print("Artifacts:", ARTIFACT_DIR)
print("Base feature order:", FEATURE_COLUMNS)


Project root: /Users/elijahzyp/Desktop/CMUSPRING2026/EPS/Project
Dataset: /Users/elijahzyp/Desktop/CMUSPRING2026/EPS/Project/data/PS_20174392719_1491204439457_log.csv
Artifacts: /Users/elijahzyp/Desktop/CMUSPRING2026/EPS/Project/artifacts
Base feature order: ['amount_scaled', 'oldbalanceOrg_scaled', 'oldbalanceDest_scaled', 'deltaOrig_scaled', 'deltaDest_scaled', 'is_transfer']


## 3. Data Loading And Sampling

We scan the full PaySim CSV in chunks, keep only `TRANSFER` and `CASH_OUT`, retain every fraud row, and downsample non-fraud rows to roughly a 10:1 non-fraud to fraud ratio.


In [3]:
def scan_selected_type_counts(data_path: Path, chunk_size: int = CHUNK_SIZE) -> tuple[int, int]:
    selected_fraud = 0
    selected_non_fraud = 0
    for chunk in pd.read_csv(data_path, usecols=["type", "isFraud"], chunksize=chunk_size):
        chunk = chunk[chunk["type"].isin(SELECTED_TYPES)]
        if chunk.empty:
            continue
        selected_fraud += int(chunk["isFraud"].sum())
        selected_non_fraud += int((chunk["isFraud"] == 0).sum())
    return selected_fraud, selected_non_fraud


def build_sample_dataframe(
    data_path: Path,
    negative_sample_prob: float,
    chunk_size: int = CHUNK_SIZE,
    random_state: int = RANDOM_STATE,
) -> pd.DataFrame:
    rng = np.random.default_rng(random_state)
    usecols = [
        "step",
        "type",
        "amount",
        "nameOrig",
        "oldbalanceOrg",
        "newbalanceOrig",
        "nameDest",
        "oldbalanceDest",
        "newbalanceDest",
        "isFraud",
        "isFlaggedFraud",
    ]

    sampled_chunks = []
    for chunk in pd.read_csv(data_path, usecols=usecols, chunksize=chunk_size):
        chunk = chunk[chunk["type"].isin(SELECTED_TYPES)].copy()
        if chunk.empty:
            continue

        fraud_df = chunk[chunk["isFraud"] == 1]
        non_fraud_df = chunk[chunk["isFraud"] == 0]
        sampled_non_fraud_df = non_fraud_df.loc[
            rng.random(len(non_fraud_df)) < negative_sample_prob
        ]
        sampled_chunks.append(pd.concat([fraud_df, sampled_non_fraud_df], ignore_index=True))

    sample_df = pd.concat(sampled_chunks, ignore_index=True)
    sample_df = sample_df.sample(frac=1.0, random_state=random_state).reset_index(drop=True)
    sample_df.insert(0, "sample_row_id", np.arange(len(sample_df), dtype=int))
    return sample_df


fraud_count, selected_non_fraud_count = scan_selected_type_counts(DATA_PATH)
target_negative_samples = NEG_POS_RATIO * fraud_count
negative_sample_prob = min(1.0, target_negative_samples / selected_non_fraud_count)

sample_df = build_sample_dataframe(DATA_PATH, negative_sample_prob=negative_sample_prob)

sample_summary_df = pd.DataFrame(
    {
        "metric": [
            "selected_fraud_rows",
            "selected_non_fraud_rows",
            "target_negative_samples",
            "negative_sample_probability",
            "sample_rows",
            "sample_fraud",
            "sample_non_fraud",
        ],
        "value": [
            fraud_count,
            selected_non_fraud_count,
            target_negative_samples,
            negative_sample_prob,
            len(sample_df),
            int(sample_df["isFraud"].sum()),
            int((sample_df["isFraud"] == 0).sum()),
        ],
    }
)

display(sample_summary_df)
display(pd.crosstab(sample_df["type"], sample_df["isFraud"]))


,metric,value
0,selected_fraud_rows,"8,213.0000"
1,selected_non_fraud_rows,"2,762,196.0000"
2,target_negative_samples,"82,130.0000"
3,negative_sample_probability,0.0297
4,sample_rows,"89,998.0000"
5,sample_fraud,"8,213.0000"
6,sample_non_fraud,"81,785.0000"


isFraud,0,1
type,,
CASH_OUT,65972,4116
TRANSFER,15813,4097


## 4. Feature Engineering

The canonical base representation remains the same:

1. `amount_scaled`
2. `oldbalanceOrg_scaled`
3. `oldbalanceDest_scaled`
4. `deltaOrig_scaled`
5. `deltaDest_scaled`
6. `is_transfer`

Raw engineered features:

- `deltaOrig = oldbalanceOrg - newbalanceOrig`
- `deltaDest = newbalanceDest - oldbalanceDest`
- `is_transfer = 1` for `TRANSFER`, else `0`


In [4]:
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["deltaOrig"] = df["oldbalanceOrg"] - df["newbalanceOrig"]
    df["deltaDest"] = df["newbalanceDest"] - df["oldbalanceDest"]
    df["is_transfer"] = (df["type"] == "TRANSFER").astype(int)
    return df


sample_df = add_features(sample_df)
display(sample_df[NUMERIC_BASE_FEATURES + BINARY_FEATURES + ["isFraud"]].head())


,amount,oldbalanceOrg,oldbalanceDest,deltaOrig,deltaDest,is_transfer,isFraud
0,"90,430.4000",0.0000,"129,644.3000",0.0000,"90,430.4000",0,0
1,"98,733.7300",0.0000,"344,035.0000",0.0000,"526,054.5200",0,0
2,"95,636.2100",367.0000,0.0000,367.0000,"95,636.2100",0,0
3,"54,294.1700",0.0000,"391,758.4300",0.0000,"54,294.1700",0,0
4,"481,486.0000","34,518.0000",0.0000,"34,518.0000","481,486.0000",1,0


## 5. Train / Validation / Test Split

We keep the same deterministic stratified split logic so that all four models are compared on the same sampled rows.


In [5]:
train_full_df, test_df = train_test_split(
    sample_df,
    test_size=TEST_SIZE,
    stratify=sample_df["isFraud"],
    random_state=RANDOM_STATE,
)

train_df, val_df = train_test_split(
    train_full_df,
    test_size=VAL_SIZE_WITHIN_TRAIN,
    stratify=train_full_df["isFraud"],
    random_state=RANDOM_STATE,
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

split_summary_df = pd.DataFrame(
    [
        {
            "split": "train",
            "rows": len(train_df),
            "fraud": int(train_df["isFraud"].sum()),
            "non_fraud": int((train_df["isFraud"] == 0).sum()),
        },
        {
            "split": "validation",
            "rows": len(val_df),
            "fraud": int(val_df["isFraud"].sum()),
            "non_fraud": int((val_df["isFraud"] == 0).sum()),
        },
        {
            "split": "test",
            "rows": len(test_df),
            "fraud": int(test_df["isFraud"].sum()),
            "non_fraud": int((test_df["isFraud"] == 0).sum()),
        },
    ]
)

display(split_summary_df)


,split,rows,fraud,non_fraud
0,train,57598,5256,52342
1,validation,14400,1314,13086
2,test,18000,1643,16357


## 6. Scaling

We fit `StandardScaler` on the training split only for the five numeric features:

- `amount`
- `oldbalanceOrg`
- `oldbalanceDest`
- `deltaOrig`
- `deltaDest`

The binary `is_transfer` feature stays unscaled.


In [6]:
def add_scaled_feature_columns(df: pd.DataFrame, scaler: StandardScaler) -> pd.DataFrame:
    df = df.copy()
    scaled_values = scaler.transform(df[NUMERIC_BASE_FEATURES])
    scaled_df = pd.DataFrame(scaled_values, columns=NUMERIC_SCALED_FEATURES, index=df.index)
    return pd.concat([df, scaled_df], axis=1)


scaler = StandardScaler()
scaler.fit(train_df[NUMERIC_BASE_FEATURES])

train_df = add_scaled_feature_columns(train_df, scaler)
val_df = add_scaled_feature_columns(val_df, scaler)
test_df = add_scaled_feature_columns(test_df, scaler)

X_train = train_df[FEATURE_COLUMNS].to_numpy(dtype=float)
y_train = train_df["isFraud"].to_numpy(dtype=int)
X_val = val_df[FEATURE_COLUMNS].to_numpy(dtype=float)
y_val = val_df["isFraud"].to_numpy(dtype=int)
X_test = test_df[FEATURE_COLUMNS].to_numpy(dtype=float)
y_test = test_df["isFraud"].to_numpy(dtype=int)

display(train_df[FEATURE_COLUMNS + ["isFraud"]].head())


,amount_scaled,oldbalanceOrg_scaled,oldbalanceDest_scaled,deltaOrig_scaled,deltaDest_scaled,is_transfer,isFraud
0,-0.3078,-0.1644,-0.3286,-0.1898,-0.2410,0,0
1,-0.1286,-0.1644,-0.0387,-0.1898,-0.0856,1,0
2,2.4160,-0.1644,0.5675,-0.1898,2.1218,1,0
3,-0.2214,-0.1612,-0.3946,-0.1854,-0.1661,0,0
4,-0.2727,-0.1544,-0.3480,-0.1761,-0.2106,0,0


## 7. Model Training

We train four models on the same base 6D representation:

- Ridge score baseline on the 6D inputs
- Logistic Regression on the 6D inputs
- LinearSVC on the 6D inputs
- Degree-2 polynomial Logistic Regression on polynomially expanded 6D inputs

For the polynomial model, we explicitly apply `PolynomialFeatures(degree=2, include_bias=False)` to the scaled 6D base inputs. With 6 base features, the expanded dimension is 27.


In [7]:
def weighted_standardize(X: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    mean = X.mean(axis=0)
    scale = X.std(axis=0)
    scale[scale == 0] = 1.0
    return (X - mean) / scale, mean, scale


def fit_ridge_score_model(
    X: np.ndarray,
    y_pm1: np.ndarray,
    alpha: float = 1.0,
) -> tuple[np.ndarray, float]:
    Xz, x_mean, x_scale = weighted_standardize(X)
    X_aug = np.column_stack([Xz, np.ones(len(Xz))])
    reg = alpha * np.eye(X_aug.shape[1])
    reg[-1, -1] = 0.0
    coef = np.linalg.solve(X_aug.T @ X_aug + reg, X_aug.T @ y_pm1.astype(float))

    weights_z = coef[:-1]
    bias_z = coef[-1]
    weights_raw = weights_z / x_scale
    bias_raw = bias_z - np.sum((weights_z * x_mean) / x_scale)
    return weights_raw, float(bias_raw)


def fit_logistic_regression(
    X: np.ndarray,
    y: np.ndarray,
    l2: float = 1.0,
    max_iter: int = 500,
) -> tuple[np.ndarray, float, object]:
    X_aug = np.column_stack([X, np.ones(len(X))])
    y = y.astype(float)

    def objective(w: np.ndarray) -> float:
        z = X_aug @ w
        p = expit(z)
        eps = 1e-12
        loss = -(y * np.log(p + eps) + (1.0 - y) * np.log(1.0 - p + eps)).mean()
        reg = 0.5 * l2 * np.sum(w[:-1] ** 2) / len(y)
        return float(loss + reg)

    def gradient(w: np.ndarray) -> np.ndarray:
        z = X_aug @ w
        p = expit(z)
        grad = (X_aug.T @ (p - y)) / len(y)
        grad[:-1] += l2 * w[:-1] / len(y)
        return grad

    init = np.zeros(X_aug.shape[1], dtype=float)
    result = minimize(
        objective,
        init,
        jac=gradient,
        method="L-BFGS-B",
        options={"maxiter": max_iter},
    )
    return result.x[:-1], float(result.x[-1]), result


ridge_w, ridge_b = fit_ridge_score_model(X_train, 2 * y_train - 1, alpha=1.0)
logreg_w, logreg_b, logreg_result = fit_logistic_regression(X_train, y_train, l2=1.0, max_iter=500)

linear_svc_model = LinearSVC(
    C=1.0,
    max_iter=5000,
    random_state=RANDOM_STATE,
    dual=False,
)
linear_svc_model.fit(X_train, y_train)

poly2_transformer = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly2_transformer.fit_transform(X_train)
X_val_poly = poly2_transformer.transform(X_val)
X_test_poly = poly2_transformer.transform(X_test)

poly2_logreg_model = LogisticRegression(
    penalty="l2",
    C=1.0,
    solver="lbfgs",
    max_iter=1000,
    random_state=RANDOM_STATE,
)
poly2_logreg_model.fit(X_train_poly, y_train)

poly2_feature_names = poly2_transformer.get_feature_names_out(FEATURE_COLUMNS)
poly2_feature_dim = int(X_train_poly.shape[1])

print("Logistic optimizer success:", bool(logreg_result.success))
print("Logistic optimizer message:", str(logreg_result.message))
print("Polynomial degree-2 expanded feature dimension:", poly2_feature_dim)
display(pd.DataFrame({"poly2_feature_name": poly2_feature_names[:15]}))


Logistic optimizer success: True
Logistic optimizer message: CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
Polynomial degree-2 expanded feature dimension: 27


,poly2_feature_name
0,amount_scaled
1,oldbalanceOrg_scaled
2,oldbalanceDest_scaled
3,deltaOrig_scaled
4,deltaDest_scaled
5,is_transfer
6,amount_scaled^2
7,amount_scaled oldbalanceOrg_scaled
8,amount_scaled oldbalanceDest_scaled
9,amount_scaled deltaOrig_scaled


## 8. Evaluation

We report precision, recall, F1, PR AUC, confusion matrix, and predicted positive counts for validation and test across all four models.

If the archived 8D artifact is available, we also compare the new 6D results against the previous 8D setup. The main interpretation focus is whether compact 6D inputs remain competitive while staying HE-friendly.


In [8]:
def evaluate_split(
    split_name: str,
    model_name: str,
    y_true: np.ndarray,
    y_pred: np.ndarray,
    y_score: np.ndarray,
) -> dict:
    return {
        "split": split_name,
        "model": model_name,
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "pr_auc": float(average_precision_score(y_true, y_score)),
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=[0, 1]).astype(int).tolist(),
        "predicted_positive": int(y_pred.sum()),
    }


def maybe_load_previous_baseline(previous_path: Path):
    if not previous_path.exists():
        return None
    with previous_path.open() as f:
        previous = json.load(f)
    return previous if "metrics" in previous else None


def summarize_comparison(previous_metrics: dict, current_metrics: dict) -> list[str]:
    comparisons = []
    for key in [("ridge_score", "test"), ("logistic_regression", "test")]:
        current = current_metrics.get(key)
        previous = previous_metrics.get(key)
        if not current or not previous:
            continue

        delta_f1 = current["f1"] - previous["f1"]
        delta_pr_auc = current["pr_auc"] - previous["pr_auc"]
        if abs(delta_f1) < 0.01 and abs(delta_pr_auc) < 0.01:
            verdict = "No material performance change."
        elif delta_f1 >= 0 and delta_pr_auc >= 0:
            verdict = "Performance improved slightly."
        else:
            verdict = "Performance changed modestly."

        model_label = "Ridge" if key[0] == "ridge_score" else "Logistic Regression"
        comparisons.append(
            f"{model_label}: F1 {previous['f1']:.4f} -> {current['f1']:.4f}, "
            f"PR AUC {previous['pr_auc']:.4f} -> {current['pr_auc']:.4f}. {verdict}"
        )
    return comparisons


ridge_val_score = X_val @ ridge_w + ridge_b
ridge_test_score = X_test @ ridge_w + ridge_b
ridge_val_pred = (ridge_val_score >= 0.0).astype(int)
ridge_test_pred = (ridge_test_score >= 0.0).astype(int)

logreg_val_logit = X_val @ logreg_w + logreg_b
logreg_test_logit = X_test @ logreg_w + logreg_b
logreg_val_prob = expit(logreg_val_logit)
logreg_test_prob = expit(logreg_test_logit)
logreg_val_pred = (logreg_val_prob >= 0.5).astype(int)
logreg_test_pred = (logreg_test_prob >= 0.5).astype(int)

linear_svc_val_score = linear_svc_model.decision_function(X_val)
linear_svc_test_score = linear_svc_model.decision_function(X_test)
linear_svc_val_pred = (linear_svc_val_score >= 0.0).astype(int)
linear_svc_test_pred = (linear_svc_test_score >= 0.0).astype(int)

poly2_val_logit = poly2_logreg_model.decision_function(X_val_poly)
poly2_test_logit = poly2_logreg_model.decision_function(X_test_poly)
poly2_val_prob = poly2_logreg_model.predict_proba(X_val_poly)[:, 1]
poly2_test_prob = poly2_logreg_model.predict_proba(X_test_poly)[:, 1]
poly2_val_pred = (poly2_val_prob >= 0.5).astype(int)
poly2_test_pred = (poly2_test_prob >= 0.5).astype(int)

metrics = [
    evaluate_split("validation", "ridge_score", y_val, ridge_val_pred, ridge_val_score),
    evaluate_split("test", "ridge_score", y_test, ridge_test_pred, ridge_test_score),
    evaluate_split("validation", "logistic_regression", y_val, logreg_val_pred, logreg_val_prob),
    evaluate_split("test", "logistic_regression", y_test, logreg_test_pred, logreg_test_prob),
    evaluate_split("validation", "linear_svc", y_val, linear_svc_val_pred, linear_svc_val_score),
    evaluate_split("test", "linear_svc", y_test, linear_svc_test_pred, linear_svc_test_score),
    evaluate_split("validation", "poly2_logistic_regression", y_val, poly2_val_pred, poly2_val_prob),
    evaluate_split("test", "poly2_logistic_regression", y_test, poly2_test_pred, poly2_test_prob),
]

metrics_df = pd.DataFrame(metrics)
model_order = [
    "ridge_score",
    "logistic_regression",
    "linear_svc",
    "poly2_logistic_regression",
]
metrics_df["model"] = pd.Categorical(metrics_df["model"], categories=model_order, ordered=True)
metrics_df["split"] = pd.Categorical(metrics_df["split"], categories=["validation", "test"], ordered=True)
metrics_df = metrics_df.sort_values(["split", "model"]).reset_index(drop=True)
display(metrics_df)

compact_comparison_df = metrics_df[
    ["split", "model", "precision", "recall", "f1", "pr_auc", "predicted_positive"]
].copy()
display(compact_comparison_df)

metrics_by_key = {(item["model"], item["split"]): item for item in metrics}
previous_summary = maybe_load_previous_baseline(PREVIOUS_8D_ARTIFACT_PATH)

if previous_summary is None:
    comparison_to_previous_8d = ["Previous 8D artifact not available in archive/. Comparison skipped."]
else:
    previous_metrics_by_key = {}
    for item in previous_summary.get("metrics", []):
        model_name = item.get("model", "")
        if model_name == "ridge_score_val":
            previous_metrics_by_key[("ridge_score", "validation")] = item
        elif model_name == "ridge_score_test":
            previous_metrics_by_key[("ridge_score", "test")] = item
        elif model_name == "logreg_val":
            previous_metrics_by_key[("logistic_regression", "validation")] = item
        elif model_name == "logreg_test":
            previous_metrics_by_key[("logistic_regression", "test")] = item
    comparison_to_previous_8d = summarize_comparison(previous_metrics_by_key, metrics_by_key)
    if not comparison_to_previous_8d:
        comparison_to_previous_8d = ["Previous 8D artifact found, but comparable metric entries were missing."]

print("Comparison to previous 8D baseline:")
for line in comparison_to_previous_8d:
    print("-", line)


,split,model,precision,recall,f1,pr_auc,confusion_matrix,predicted_positive
0,validation,ridge_score,1.0000,0.1781,0.3023,0.7480,"[[13086, 0], [1080, 234]]",234
1,validation,logistic_regression,0.9333,0.6819,0.7880,0.8977,"[[13022, 64], [418, 896]]",960
2,validation,linear_svc,0.9373,0.6256,0.7503,0.8721,"[[13031, 55], [492, 822]]",877
3,validation,poly2_logistic_regression,0.9621,0.7922,0.8689,0.9353,"[[13045, 41], [273, 1041]]",1082
4,test,ridge_score,1.0000,0.1862,0.3140,0.7335,"[[16357, 0], [1337, 306]]",306
5,test,logistic_regression,0.9214,0.6853,0.7860,0.8936,"[[16261, 96], [517, 1126]]",1222
6,test,linear_svc,0.9218,0.6239,0.7441,0.8670,"[[16270, 87], [618, 1025]]",1112
7,test,poly2_logistic_regression,0.9526,0.7833,0.8597,0.9336,"[[16293, 64], [356, 1287]]",1351


,split,model,precision,recall,f1,pr_auc,predicted_positive
0,validation,ridge_score,1.0000,0.1781,0.3023,0.7480,234
1,validation,logistic_regression,0.9333,0.6819,0.7880,0.8977,960
2,validation,linear_svc,0.9373,0.6256,0.7503,0.8721,877
3,validation,poly2_logistic_regression,0.9621,0.7922,0.8689,0.9353,1082
4,test,ridge_score,1.0000,0.1862,0.3140,0.7335,306
5,test,logistic_regression,0.9214,0.6853,0.7860,0.8936,1222
6,test,linear_svc,0.9218,0.6239,0.7441,0.8670,1112
7,test,poly2_logistic_regression,0.9526,0.7833,0.8597,0.9336,1351


Comparison to previous 8D baseline:
- Ridge: F1 0.3226 -> 0.3140, PR AUC 0.7414 -> 0.7335. No material performance change.
- Logistic Regression: F1 0.7881 -> 0.7860, PR AUC 0.8989 -> 0.8936. No material performance change.


## 9. Artifact Export

We preserve the current canonical 6D artifacts and extend them with the new models:

- `baseline_6d_summary.json`
- `baseline_model_comparison_summary.json`
- `feature_order_6d.json`
- `baseline_6d_scaler.pkl`
- `baseline_6d_ridge.pkl`
- `baseline_6d_logreg.pkl`
- `linear_svc_model.pkl`
- `poly2_logreg_model.pkl`
- `poly2_transformer.pkl`
- `he_eval_scaled_features_6d.csv`
- `he_eval_meta_6d.csv`
- `he_eval_plaintext_scores_6d.csv`

For the polynomial model, the current HE design uses **pre-expanded plaintext polynomial features encrypted afterward**. This avoids encrypted-space polynomial term construction in this pass.


In [9]:
def choose_he_eval_subset(
    df: pd.DataFrame,
    fraud_target: int = 32,
    non_fraud_target: int = 96,
    random_state: int = RANDOM_STATE,
) -> pd.DataFrame:
    rng = np.random.default_rng(random_state)
    fraud_idx = df.index[df["isFraud"] == 1].to_numpy()
    non_fraud_idx = df.index[df["isFraud"] == 0].to_numpy()

    fraud_take = min(fraud_target, len(fraud_idx))
    non_fraud_take = min(non_fraud_target, len(non_fraud_idx))

    fraud_sel = rng.choice(fraud_idx, size=fraud_take, replace=False) if fraud_take else np.array([], dtype=int)
    non_fraud_sel = (
        rng.choice(non_fraud_idx, size=non_fraud_take, replace=False) if non_fraud_take else np.array([], dtype=int)
    )

    selected = np.concatenate([fraud_sel, non_fraud_sel])
    rng.shuffle(selected)
    return df.loc[selected].reset_index(drop=True)


feature_order = {
    "feature_columns": FEATURE_COLUMNS,
    "numeric_base_features": NUMERIC_BASE_FEATURES,
    "binary_features": BINARY_FEATURES,
}

model_metadata = [
    {
        "model_name": "ridge_score",
        "feature_representation": "base_6d_scaled",
        "score_type": "decision_function",
        "he_ready": True,
        "he_strategy": "encrypt base 6D scaled features and evaluate a weighted sum",
        "coefficient_shape": list(np.asarray(ridge_w).shape),
        "intercept_shape": [],
    },
    {
        "model_name": "logistic_regression",
        "feature_representation": "base_6d_scaled",
        "score_type": "raw_logit",
        "he_ready": True,
        "he_strategy": "encrypt base 6D scaled features and evaluate a weighted sum",
        "coefficient_shape": list(np.asarray(logreg_w).shape),
        "intercept_shape": [],
    },
    {
        "model_name": "linear_svc",
        "feature_representation": "base_6d_scaled",
        "score_type": "decision_function",
        "he_ready": True,
        "he_strategy": "encrypt base 6D scaled features and evaluate a weighted sum",
        "coefficient_shape": list(linear_svc_model.coef_.reshape(-1).shape),
        "intercept_shape": list(linear_svc_model.intercept_.shape),
    },
    {
        "model_name": "poly2_logistic_regression",
        "feature_representation": "poly2_expanded_from_base_6d",
        "score_type": "raw_linear_score_on_poly2_features",
        "he_ready": True,
        "he_strategy": "pre-expand plaintext base 6D features with PolynomialFeatures, then encrypt the expanded vector",
        "coefficient_shape": list(poly2_logreg_model.coef_.reshape(-1).shape),
        "intercept_shape": list(poly2_logreg_model.intercept_.shape),
        "base_feature_order": FEATURE_COLUMNS,
        "expanded_feature_dimension": poly2_feature_dim,
    },
]

he_eval_df = choose_he_eval_subset(test_df)
he_eval_scaled_features_df = he_eval_df[FEATURE_COLUMNS].copy()
he_eval_poly2_features = poly2_transformer.transform(he_eval_scaled_features_df.to_numpy(dtype=float))
he_eval_X = he_eval_scaled_features_df.to_numpy(dtype=float)
he_eval_meta_df = he_eval_df[["sample_row_id", "step", "type", "isFraud"]].copy()
he_eval_scores_df = pd.DataFrame(
    {
        "sample_row_id": he_eval_df["sample_row_id"],
        "isFraud": he_eval_df["isFraud"],
        "ridge_raw_score": he_eval_X @ ridge_w + ridge_b,
        "logistic_raw_logit": he_eval_X @ logreg_w + logreg_b,
        "logistic_probability": expit(he_eval_X @ logreg_w + logreg_b),
        "linear_svc_decision": linear_svc_model.decision_function(he_eval_X),
        "poly2_logreg_raw_score": poly2_logreg_model.decision_function(he_eval_poly2_features),
        "poly2_logreg_probability": poly2_logreg_model.predict_proba(he_eval_poly2_features)[:, 1],
    }
)

detailed_summary = {
    "feature_order": feature_order,
    "scaler": {
        "mean": scaler.mean_.tolist(),
        "scale": scaler.scale_.tolist(),
    },
    "ridge_score": {
        "weights": ridge_w.tolist(),
        "bias": float(ridge_b),
    },
    "logistic_regression": {
        "weights": logreg_w.tolist(),
        "bias": float(logreg_b),
        "optimizer_success": bool(logreg_result.success),
        "optimizer_message": str(logreg_result.message),
    },
    "linear_svc": {
        "weights": linear_svc_model.coef_.reshape(-1).tolist(),
        "bias": float(linear_svc_model.intercept_[0]),
        "classes": linear_svc_model.classes_.tolist(),
    },
    "poly2_logistic_regression": {
        "weights": poly2_logreg_model.coef_.reshape(-1).tolist(),
        "bias": float(poly2_logreg_model.intercept_[0]),
        "expanded_feature_dimension": poly2_feature_dim,
        "optimizer_iterations": poly2_logreg_model.n_iter_.tolist(),
    },
    "poly2_features": {
        "degree": 2,
        "include_bias": False,
        "expanded_feature_dimension": poly2_feature_dim,
        "base_feature_order": FEATURE_COLUMNS,
        "feature_names": poly2_feature_names.tolist(),
        "he_strategy": "pre-expanded plaintext features encrypted afterward",
    },
    "sample_summary": {
        "rows": int(len(sample_df)),
        "fraud": int(sample_df["isFraud"].sum()),
        "non_fraud": int((sample_df["isFraud"] == 0).sum()),
        "types": {str(key): int(value) for key, value in sample_df["type"].value_counts().to_dict().items()},
    },
    "split_summary": {
        "train_rows": int(len(train_df)),
        "validation_rows": int(len(val_df)),
        "test_rows": int(len(test_df)),
        "train_fraud": int(train_df["isFraud"].sum()),
        "validation_fraud": int(val_df["isFraud"].sum()),
        "test_fraud": int(test_df["isFraud"].sum()),
    },
    "model_metadata": model_metadata,
    "metrics": metrics,
    "comparison_to_previous_8d": comparison_to_previous_8d,
    "he_eval_subset": {
        "rows": int(len(he_eval_df)),
        "fraud": int(he_eval_df["isFraud"].sum()),
        "non_fraud": int((he_eval_df["isFraud"] == 0).sum()),
        "feature_csv": str(HE_FEATURES_PATH),
        "meta_csv": str(HE_META_PATH),
        "plaintext_scores_csv": str(HE_SCORES_PATH),
    },
}

compact_summary = {
    "base_feature_order": FEATURE_COLUMNS,
    "poly2_expanded_feature_dimension": poly2_feature_dim,
    "he_strategy": {
        "linear_models": "encrypt base 6D scaled features and evaluate weighted sums",
        "poly2_logistic_regression": "pre-expand plaintext polynomial features, then encrypt the expanded vector",
    },
    "models": model_metadata,
    "metrics": metrics,
}

FEATURE_ORDER_JSON_PATH.write_text(json.dumps(feature_order, indent=2))
SUMMARY_JSON_PATH.write_text(json.dumps(detailed_summary, indent=2))
COMPARISON_JSON_PATH.write_text(json.dumps(compact_summary, indent=2))

with SCALER_PICKLE_PATH.open("wb") as f:
    pickle.dump(scaler, f)
with RIDGE_PICKLE_PATH.open("wb") as f:
    pickle.dump({"weights": ridge_w, "bias": ridge_b}, f)
with LOGREG_PICKLE_PATH.open("wb") as f:
    pickle.dump({"weights": logreg_w, "bias": logreg_b}, f)
with LINEAR_SVC_PICKLE_PATH.open("wb") as f:
    pickle.dump(linear_svc_model, f)
with POLY2_LOGREG_PICKLE_PATH.open("wb") as f:
    pickle.dump(poly2_logreg_model, f)
with POLY2_TRANSFORMER_PICKLE_PATH.open("wb") as f:
    pickle.dump(poly2_transformer, f)

he_eval_scaled_features_df.to_csv(HE_FEATURES_PATH, index=False)
he_eval_meta_df.to_csv(HE_META_PATH, index=False)
he_eval_scores_df.to_csv(HE_SCORES_PATH, index=False)

exported_paths = [
    SUMMARY_JSON_PATH,
    COMPARISON_JSON_PATH,
    FEATURE_ORDER_JSON_PATH,
    SCALER_PICKLE_PATH,
    RIDGE_PICKLE_PATH,
    LOGREG_PICKLE_PATH,
    LINEAR_SVC_PICKLE_PATH,
    POLY2_LOGREG_PICKLE_PATH,
    POLY2_TRANSFORMER_PICKLE_PATH,
    HE_FEATURES_PATH,
    HE_META_PATH,
    HE_SCORES_PATH,
]

print("Exported final files:")
for path in exported_paths:
    print("-", path)


Exported final files:
- /Users/elijahzyp/Desktop/CMUSPRING2026/EPS/Project/artifacts/baseline_6d_summary.json
- /Users/elijahzyp/Desktop/CMUSPRING2026/EPS/Project/artifacts/baseline_model_comparison_summary.json
- /Users/elijahzyp/Desktop/CMUSPRING2026/EPS/Project/artifacts/feature_order_6d.json
- /Users/elijahzyp/Desktop/CMUSPRING2026/EPS/Project/artifacts/baseline_6d_scaler.pkl
- /Users/elijahzyp/Desktop/CMUSPRING2026/EPS/Project/artifacts/baseline_6d_ridge.pkl
- /Users/elijahzyp/Desktop/CMUSPRING2026/EPS/Project/artifacts/baseline_6d_logreg.pkl
- /Users/elijahzyp/Desktop/CMUSPRING2026/EPS/Project/artifacts/linear_svc_model.pkl
- /Users/elijahzyp/Desktop/CMUSPRING2026/EPS/Project/artifacts/poly2_logreg_model.pkl
- /Users/elijahzyp/Desktop/CMUSPRING2026/EPS/Project/artifacts/poly2_transformer.pkl
- /Users/elijahzyp/Desktop/CMUSPRING2026/EPS/Project/artifacts/he_eval_scaled_features_6d.csv
- /Users/elijahzyp/Desktop/CMUSPRING2026/EPS/Project/artifacts/he_eval_meta_6d.csv
- /Users/elija

## 10. HE Evaluation Subset Explanation

The HE subset files play the following roles:

- `he_eval_scaled_features_6d.csv` contains the final scaled 6D feature vectors in exact base model input order
- `he_eval_meta_6d.csv` contains labels and lightweight metadata for the same rows
- `he_eval_plaintext_scores_6d.csv` contains plaintext reference scores for all exported models

For the polynomial model, the HE notebook will transform the base 6D features into polynomial degree-2 features in plaintext first, then encrypt the expanded feature vector for score computation.


In [10]:
final_summary_lines = [
    f"Sample rows: {len(sample_df):,}",
    f"Fraud rows: {int(sample_df['isFraud'].sum()):,}",
    f"Non-fraud rows: {int((sample_df['isFraud'] == 0).sum()):,}",
    f"Final 6D features: {FEATURE_COLUMNS}",
    f"Polynomial degree-2 expanded dimension: {poly2_feature_dim}",
    "",
    "Validation and test metrics:",
]

for metric in metrics_df.to_dict(orient="records"):
    final_summary_lines.append(
        f"{metric['split']} {metric['model']}: "
        f"precision={metric['precision']:.4f}, "
        f"recall={metric['recall']:.4f}, "
        f"f1={metric['f1']:.4f}, "
        f"pr_auc={metric['pr_auc']:.4f}, "
        f"predicted_positive={metric['predicted_positive']}, "
        f"confusion_matrix={metric['confusion_matrix']}"
    )

final_summary_lines.extend(
    [
        "",
        "Exported files:",
        *[f"- {path}" for path in exported_paths],
        "",
        "Ready for next HE stage: yes. The canonical artifacts now cover ridge, logistic regression, LinearSVC, and degree-2 polynomial logistic regression.",
    ]
)

print("\n".join(final_summary_lines))


Sample rows: 89,998
Fraud rows: 8,213
Non-fraud rows: 81,785
Final 6D features: ['amount_scaled', 'oldbalanceOrg_scaled', 'oldbalanceDest_scaled', 'deltaOrig_scaled', 'deltaDest_scaled', 'is_transfer']
Polynomial degree-2 expanded dimension: 27

Validation and test metrics:
validation ridge_score: precision=1.0000, recall=0.1781, f1=0.3023, pr_auc=0.7480, predicted_positive=234, confusion_matrix=[[13086, 0], [1080, 234]]
validation logistic_regression: precision=0.9333, recall=0.6819, f1=0.7880, pr_auc=0.8977, predicted_positive=960, confusion_matrix=[[13022, 64], [418, 896]]
validation linear_svc: precision=0.9373, recall=0.6256, f1=0.7503, pr_auc=0.8721, predicted_positive=877, confusion_matrix=[[13031, 55], [492, 822]]
validation poly2_logistic_regression: precision=0.9621, recall=0.7922, f1=0.8689, pr_auc=0.9353, predicted_positive=1082, confusion_matrix=[[13045, 41], [273, 1041]]
test ridge_score: precision=1.0000, recall=0.1862, f1=0.3140, pr_auc=0.7335, predicted_positive=306, c